# Random-Forest Magnitude Prediction

## What this notebook does
This notebook loops through every CSV file in a Google Drive input folder, removes highly correlated predictors, tunes a random-forest regression model with Optuna and five-fold cross-validation, evaluates it on held-out test data, and saves one results table per dataset.

## Required inputs
- Google Colab with access to Google Drive.
- One or more CSV files stored directly in `input_folder`.
- Each CSV must contain the columns expected by the preprocessing section described below.

## Outputs
- One `<dataset>_RF_results.csv` file per input CSV, containing training and test R-squared, MAE, RMSE, MAPE, and the best hyperparameters.
- Cross-validation and processing progress printed in the notebook output.

## Settings a new user should review
- `input_folder`: change this to the Google Drive folder containing the input CSV files.
- `output_folder`: change this to the folder where result CSV files should be written.
- `df.drop("site_id", axis=1)`: edit or remove this line if your identifier column has a different name or is absent.
- `df = df[df.columns[1:]]`: edit or remove this line if the first remaining column should not be discarded.
- The correlation threshold (`0.9`), split proportions, random seed, and number of Optuna trials can be changed if required.

## Important data assumptions
- After `site_id` is removed, the script discards the first remaining column.
- The first column left after those removals is treated as the response variable; all later columns are predictors.
- The input folder should contain only CSV files that follow this same structure, because every CSV in the folder is processed.

Run the notebook from top to bottom in Google Colab. The progress messages explain what is happening and where outputs are written.

## 1. Install required packages
Run this cell once at the start of a fresh Colab session.

In [ ]:
# Install Packages
!pip install keras-tuner tensorflow pandas numpy scikit-learn shap
!pip install optuna --quiet


## 2. Import packages

In [ ]:
# Import packages
import pandas as pd
import numpy as np
import optuna
import os
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error, mean_absolute_percentage_error, make_scorer
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import matplotlib.pyplot as plt
from google.colab import drive
from optuna.pruners import MedianPruner

## 3. Mount Google Drive and set directories
**Edit `input_folder` and `output_folder` in the next cell before running the notebook on a new system.**

In [ ]:
# Mount Google Drive so the notebook can read inputs and save outputs
drive.mount('/content/drive')

# User configuration: input and output directories
input_folder = '/content/drive/My Drive/machine_learning_colab/'
output_folder = '/content/drive/My Drive/machine_learning_colab/outputs_new/'
os.makedirs(output_folder, exist_ok = True)

## 4. Process each dataset, tune the model, and save results

In [ ]:
for file_name in os.listdir(input_folder):
    if file_name.endswith('.csv'):
        dataset_path = os.path.join(input_folder, file_name)
        dataset_name = file_name.replace('.csv', '')
        print(f"\nStarting random-forest modelling for dataset: {dataset_name}")

        # load dataset
        df = pd.read_csv(dataset_path)
        print(f"Loaded {len(df):,} rows and {len(df.columns):,} columns from {file_name}.")
        df = df.drop('site_id', axis=1)
        df = df[df.columns[1:]]

        # removal of variables with high correlation
        corr_matrix = df.iloc[:, 1:].corr().abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        to_drop = [column for column in upper.columns if any(upper[column] > 0.9)]
        df = df.drop(columns=to_drop)
        print(f"Removed {len(to_drop)} predictors with absolute pairwise correlation above 0.90.")

        # Separate response and predictors
        Y = df.iloc[:, 0].values.reshape(-1, 1)
        X = df.iloc[:, 1:]
        feature_names = X.columns.tolist()

        # Remove rows where Y has NaN
        mask = ~np.isnan(Y).flatten()
        X = X[mask]
        Y = Y[mask]

        # standardization
        scaler_x = StandardScaler()
        X_scaled = scaler_x.fit_transform(X)

        # Split data into test/train
        X_train, X_test, Y_train, Y_test = train_test_split(X_scaled, Y, test_size=0.2, random_state=42)

        # Optuna objective function
        def objective(trial):
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 100, 800),
                'max_depth': trial.suggest_int('max_depth', 10, 50),
                'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
                'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
                'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
            }
            model = RandomForestRegressor(random_state=42, n_jobs=-1, **params)

            # Use 5-fold CV on training data
            neg_mse_scores = cross_val_score(model, X_train, Y_train.ravel(), scoring='neg_mean_squared_error', cv=5)
            rmse_scores = np.sqrt(-neg_mse_scores)

            return np.mean(rmse_scores)

        # Add early stopping pruner
        pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=5, interval_steps=1)

        # Run hyperparameter optimization
        print("Starting Optuna optimization with 20 trials. This may take some time for larger datasets.")
        study = optuna.create_study(direction='minimize', pruner=pruner)
        study.optimize(objective, n_trials=20, show_progress_bar=True)

        # best parameters
        best_params = study.best_trial.params
        print(f"Optimization finished. Best parameters: {best_params}")

        # Define RF model with best parameters
        best_rf = RandomForestRegressor(random_state=42, n_jobs=-1, **best_params)
        best_rf.fit(X_train, Y_train)
        train_r2 = best_rf.score(X_train, Y_train)

        # Using test data to evaluate model predictions
        test_preds = best_rf.predict(X_test)

        def evaluate(y_true, y_pred, label='Test'):
            test_r2 = r2_score(y_true, y_pred)
            mae = mean_absolute_error(y_true, y_pred)
            rmse = np.sqrt(mean_squared_error(y_true, y_pred))
            mape = mean_absolute_percentage_error(y_true, y_pred)
            return test_r2, mae, rmse, mape

        test_r2, mae, rmse, mape = evaluate(Y_test, test_preds, label='Test')

        # Save results to CSV
        results_df = pd.DataFrame([{
            "Dataset": dataset_name,
            "Train_R2": train_r2,
            "Test_R2": test_r2,
            "MAE": mae,
            "RMSE": rmse,
            "MAPE": mape,
            "Best_Params": best_params
        }])

        output_file = os.path.join(output_folder, f"{dataset_name}_RF_results.csv")
        results_df.to_csv(output_file, index=False)
        print(f"Finished {dataset_name}. Results were saved to: {output_file}")